# Caso C · 02 ETL bronce → plata HVAC + etiquetas en captia_fault_labels

> _Tutorial · Caso de uso: **C — Anomalías HVAC** · Capa Medallion: **bronce → plata** · Spec: `docs/specs/synthetic-bms/02-domain-spec.md`_

Material docente del proyecto **CAPTIA Synthetic Data BMS** — IES Dr. Lluís Simarro,
Curso de Especialización IA & Big Data 2025-2026.


## 1. Objetivo

Mapear LBNL FDD a CAPTIA, generar line protocol para `temperature_supply`, `temperature_return`, `valve_control`, `fan_speed_01_state` y, por separado, los eventos de fallo en `captia_fault_labels`.


## 2. Qué se aprende

- Routing on-change vs continuo.
- Por qué las etiquetas no van con la telemetría.
- Cómo emitir un evento `active=1` al inicio y `active=0` al fin.


## 3. Contexto del caso de uso

El equipo C necesita preservar la trazabilidad: dado un timestamp puede responder ¿hay fallo activo aquí? sin contaminar `captia_point`.


## 4. Relación con CENTINELA+

Mismo enfoque que producción real: telemetría limpia + labels separados.


## 5. Relación con Medallion

Bronce → plata + etiquetas en bucket `state_events` separado.


## 6. Datos de entrada

`lbnl_fdd_rtu_mock.csv`.


## 7. Setup y variables de entorno

Cargamos las variables de entorno (`.env`), inicializamos `numpy` con `seed=42` y aplicamos el estilo de plotting compartido. Los helpers viven en `notebooks/_common/`.


In [1]:
# Setup canónico — todos los notebooks didácticos lo usan
from __future__ import annotations

import os
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT.name and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from notebooks._common.captia_schema import (
    CANONICAL_TAGS, MEASUREMENT_TELEMETRY, MEASUREMENT_FAULT_LABELS,
    DEFAULT_BUCKET_RETENTIONS, KNOWN_VARIABLES,
    build_topic, build_line_protocol, validate_canonical_tags,
)
from notebooks._common.connection import load_env, get_influx_client, get_default_bucket
from notebooks._common.plotting import setup_default_style, plot_timeseries, plot_distribution
import notebooks._common.synthetic_mocks as mocks

SEED = 42
rng = np.random.default_rng(SEED)
setup_default_style()
load_env()
print(f"ROOT={ROOT}, SEED={SEED}, default_bucket={get_default_bucket()}")


ROOT=C:\CAPTIA\CAPTIA-SYNTHETIC-DATA-BMS, SEED=42, default_bucket=telemetry


## 8. Schema CAPTIA esperado

Para etiquetas:
```
captia_fault_labels,captia_env=dev,domain_id=hvac_system,site_id=lbnl_building59,asset_id=RTU_01,fault_type=valve_stuck active=1.0i,severity=0.74 <ts>
captia_fault_labels,...,fault_type=valve_stuck active=0.0i <ts>
```


## 9. Carga de datos o mock

Cargamos y agrupamos episodios.


In [2]:
csv_path = ROOT / "notebooks" / "_data" / "lbnl_fdd_rtu_mock.csv"
df = pd.read_csv(csv_path, comment="#", parse_dates=["timestamp"]).sort_values("timestamp").reset_index(drop=True)
df.head()


,timestamp,OA_TEMP,SA_TEMP,RA_TEMP,CCV,FAN_STATE,OCCU_MOD,is_fault,fault_type
0,2024-06-01 00:00:00+00:00,22.15,16.0,22.98,0,0,0,0,normal
1,2024-06-01 00:01:00+00:00,21.51,16.0,22.97,0,0,0,0,normal
2,2024-06-01 00:02:00+00:00,22.45,16.0,23.17,0,0,0,0,normal
3,2024-06-01 00:03:00+00:00,22.57,16.0,22.30,0,0,0,0,normal
4,2024-06-01 00:04:00+00:00,21.16,16.0,23.04,0,0,0,0,normal


## 10. Exploración paso a paso

Detectamos los episodios de fallo (transiciones).


In [3]:
df["episode_start"] = (df["fault_type"] != df["fault_type"].shift(fill_value="normal"))
episodes = df.loc[df["episode_start"]].copy()
print("Total transiciones:", len(episodes))
episodes.head()


Total transiciones: 18


,timestamp,OA_TEMP,SA_TEMP,RA_TEMP,CCV,FAN_STATE,OCCU_MOD,is_fault,fault_type,episode_start
5859,2024-06-05 01:39:00+00:00,26.01,22.87,23.37,0,0,0,1,refrigerant_low,True
6039,2024-06-05 04:39:00+00:00,29.88,16.00,22.92,0,0,0,0,normal,True
6901,2024-06-05 19:01:00+00:00,14.96,23.00,23.20,1,0,0,1,valve_stuck,True
7021,2024-06-05 21:01:00+00:00,16.98,16.00,23.24,0,0,0,0,normal,True
7758,2024-06-06 09:18:00+00:00,26.92,16.23,23.49,1,1,1,1,sensor_drift,True


## 11. Transformación bronce → plata

Generamos line protocol para telemetría continua y para etiquetas.


In [4]:
out_dir = ROOT / "output" / "case_C"
out_dir.mkdir(parents=True, exist_ok=True)

VARMAP = {
    "SA_TEMP": "temperature_supply",
    "RA_TEMP": "temperature_return",
    "OA_TEMP": "temperature_outdoor",
}
def to_lp_telemetry(row, csv_col, captia_var):
    ts_ns = int(pd.Timestamp(row["timestamp"]).value)
    return build_line_protocol(
        measurement=MEASUREMENT_TELEMETRY,
        tags={"captia_env": "dev", "domain_id": "hvac_system",
              "site_id": "lbnl_building59", "asset_id": "RTU_01",
              "variable": captia_var},
        fields={"value": float(row[csv_col])},
        timestamp_ns=ts_ns,
    )

# Solo primeras 500 filas para clase
sample = df.head(500)
lines_telem = [to_lp_telemetry(r, c, v) for c, v in VARMAP.items() for _, r in sample.iterrows()]
(out_dir / "hvac_telemetry.lp").write_text("\n".join(lines_telem) + "\n", encoding="utf-8")
print(f"Telemetry: {len(lines_telem)} líneas")


Telemetry: 1500 líneas


## 12. Construcción de capa oro

Para etiquetas: emitimos `active=1` al iniciar episodio y `active=0` al finalizar.


In [5]:
labels_lp = []
fault_runs = (df["fault_type"] != df["fault_type"].shift()).cumsum()
for run, grp in df.groupby(fault_runs):
    ftype = grp["fault_type"].iloc[0]
    if ftype == "normal":
        continue
    start_ts = int(pd.Timestamp(grp["timestamp"].iloc[0]).value)
    end_ts = int(pd.Timestamp(grp["timestamp"].iloc[-1]).value)
    labels_lp.append(
        f"captia_fault_labels,captia_env=dev,domain_id=hvac_system,site_id=lbnl_building59,asset_id=RTU_01,fault_type={ftype} "
        f"active=1i,severity=0.74 {start_ts}"
    )
    labels_lp.append(
        f"captia_fault_labels,captia_env=dev,domain_id=hvac_system,site_id=lbnl_building59,asset_id=RTU_01,fault_type={ftype} "
        f"active=0i {end_ts}"
    )
(out_dir / "hvac_fault_labels.lp").write_text("\n".join(labels_lp) + "\n", encoding="utf-8")
print(f"Etiquetas: {len(labels_lp)} eventos (start+end)")


Etiquetas: 18 eventos (start+end)


## 13. Visualizaciones explicativas

Bar chart episodios por tipo.


In [6]:
runs_summary = (
    df.assign(run=fault_runs)
      .query("fault_type != 'normal'")
      .groupby(["run", "fault_type"]).size()
      .reset_index(name="duration_min")
)
runs_summary["fault_type"].value_counts().plot.bar(color="#9C27B0")
plt.title("Episodios por tipo de fallo")
plt.tight_layout()


## 14. Validaciones

Etiquetas no contaminan la telemetría (measurement diferente).


In [7]:
assert "captia_fault_labels" in labels_lp[0]
assert "captia_point" not in labels_lp[0]
print("Schema OK — etiquetas separadas")


Schema OK — etiquetas separadas


## 15. Errores comunes

1. Mezclar `is_fault` como tag de `captia_point` (rompe cardinalidad).
2. Olvidar emitir el `active=0` al final (queda fallo abierto eternamente).
3. No usar `fault_type` como tag — perdemos granularidad.


## 16. Ejercicios propuestos

1. Añade severity proporcional a la duración del episodio.
2. Implementa solapamiento: dos fallos simultáneos.
3. Calcula MTBF (Mean Time Between Failures).


## 17. Cómo se reutiliza con datos reales

Para fallos reales del IES Simarro, la fuente serían tickets de mantenimiento. Conviértelos a este formato y pásalos por el mismo ETL.


## 18. Resumen final y próximos pasos

Recuerda los conceptos principales del notebook y enlaza al siguiente paso.

- Siguiente notebook: `03_case_C_hvac_anomaly_detection/03_features_anomalias_hvac.ipynb`.
- Documento web del caso: `docs/contracts/medallion-layers.md`.


## 19. Marco teórico (nivel doctoral)

### Isolation Forest (Liu et al. 2008)

Score basado en la profundidad media $E[h(x)]$ del path al aislar $x$ en
árboles binarios construidos por particiones aleatorias:

$$
s(x, n) = 2^{-\frac{E[h(x)]}{c(n)}}
$$

con $c(n) = 2 H(n-1) - 2(n-1)/n$ y $H(i)$ harmonic number. Anomalía si
$s(x) \to 1$, normal si $s(x) \to 0.5$.

### Autoencoder (Hinton & Salakhutdinov 2006)

$$
\hat{x} = D(E(x)), \quad E: \mathbb{R}^d \to \mathbb{R}^k, \quad D: \mathbb{R}^k \to \mathbb{R}^d, \quad k \ll d
$$

con $k = 8$ neuronas en bottleneck para $d = 24$ features. Score:

$$
e(x) = \| x - \hat{x} \|_2^2, \quad \theta = \mu_e + 3\sigma_e
$$

### Catálogo de fallos (ADR-010)

| Tipo | Variable afectada | Signature |
|---|---|---|
| `sensor_drift` | `temperature_01` | drift lineal $+0.5$ °C/h |
| `valve_stuck` | `valve_state`, $T_{int}$ | $\Delta T \to 0$ tras setpoint change |
| `fan_failure` | `fan_speed_01_state`, $T_{supply}$ | $\dot V \to 0$, $T_{supply} \to T_{coil}$ |
| `refrigerant_low` | $T_{supply} - T_{return}$ | $\Delta T_{cool}$ cae 50 % |

### Métricas

$$
\text{F1} = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}, \quad
\text{TPR}@1\%\text{FPR} = \text{Recall} \mid \text{FPR} \leq 0.01
$$

Objetivos: $\text{F1} \geq 0.85$, $\text{TPR}@1\%\text{FPR} \geq 0.7$.


## 20. Visión corporativa CAPTIA

### Propuesta de valor

Detectar fallos HVAC antes de que se manifiesten en quejas de alumnos o averías catastróficas tiene **doble valor**: ahorro operativo (mantenimiento predictivo en lugar de reactivo) y diferenciador comercial frente a competidores BMS sin IA.

### ROI estimado

| Concepto | Valor anual |
|---|---|
| Detección temprana avería catastrófica (~2/año × 3 500 €) | +7 000 € |
| Reducción downtime (2 h × 200 días) | +800 € |
| **Bruto** | **+7 800 €/año** |
| Coste integración | -2 000 € one-time |
| **Payback** | **~3 meses** |

### Riesgos y mitigaciones

- False positives → fatiga de alarmas. Tunear umbral con percentil 99.
- Drift en HVAC envejecido: incluir age-feature.


## 21. Bibliografía y referencias

- Liu, F. T., Ting, K. M. & Zhou, Z.-H. (2008). *Isolation Forest*. ICDM '08.
- Hinton, G. & Salakhutdinov, R. (2006). *Reducing the Dimensionality of Data with Neural Networks*. Science 313(5786).
- Granderson, J. et al. (2020). *Building Fault Detection Data to Aid Diagnostic Algorithm Creation and Performance Testing*. Scientific Data 7.
- ASHRAE (2021). *Guideline 36-2021 — High-Performance Sequences of Operation for HVAC Systems*.
